In [ ]:
# Jupyterでそのまま実行OK
from pathlib import Path
import json
import copy

# === 入出力設定 ===
src = Path(r"E:\permian_basin\sample.json")
out_dir = src.parent / "out_prof_ch4"
out_dir.mkdir(exist_ok=True)

# 作りたいPROFILEの定数値（11個）
targets = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0 , 2.1, 2.2, 2.3, 2.4, 2.5, 2.6, 2.7, 2.8, 2.9, 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 4.0, 4.1, 4.2, 4.3, 4.4, 4.5, 4.6, 4.7, 4.8, 4.9, 5.0]

def csv_name_from_val(v: float) -> str:
    # 小数2桁で固定（例: 1.6 -> "1.60"）
    return f"ch_{v:.2f}.csv"

def rewrite_prof_ch4(node: object, new_val: float, stats: dict):
    """
    JSONを再帰探索して "TYPE":"PROF_CH4" の辞書だけ編集：
      - PROFILE: 中の 1.8（数値/文字列）を new_val に置換。
                 もし全要素が 1.8 なら丸ごと new_val で埋め直し。
      - CSVPRNT: "ch_<new_val(小数2桁)>.csv" に設定（必ず文字列）
    """
    if isinstance(node, dict):
        is_prof = node.get("TYPE") == "PROF_CH4"

        # 先に子を処理（CSVPRNTはこのdictで上書きするので潜らない）
        for k, v in list(node.items()):
            if k == "CSVPRNT":
                continue
            node[k] = rewrite_prof_ch4(v, new_val, stats)

        if is_prof:
            stats["blocks"] += 1

            # PROFILE処理
            if "PROFILE" in node and isinstance(node["PROFILE"], list):
                prof = node["PROFILE"]
                replaced = []
                all18 = len(prof) > 0
                changed_cnt = 0
                for x in prof:
                    try:
                        xv = float(x)
                        if abs(xv - 1.8) < 1e-12:
                            replaced.append(new_val)
                            changed_cnt += 1
                        else:
                            replaced.append(x)
                            if abs(xv - 1.8) >= 1e-12:
                                all18 = False
                    except Exception:
                        replaced.append(x)
                        all18 = False

                # 全部1.8なら丸ごと new_val で置換
                if all18:
                    replaced = [new_val] * len(prof)
                    changed_cnt = len(prof)

                node["PROFILE"] = replaced
                stats["values_replaced"] += changed_cnt

            # CSVPRNTはPROFILEの値に合わせて必ず文字列で更新
            node["CSVPRNT"] = csv_name_from_val(new_val)

        return node

    elif isinstance(node, list):
        return [rewrite_prof_ch4(v, new_val, stats) for v in node]
    else:
        return node


# === 実行 ===
with src.open("r", encoding="utf-8") as f:
    original = json.load(f)

written = []
for val in targets:
    data = copy.deepcopy(original)
    stats = {"blocks": 0, "values_replaced": 0}
    data = rewrite_prof_ch4(data, val, stats)

    out_path = out_dir / f"{src.stem}_PROF_CH4_{val:.2f}.json"
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    written.append((out_path, stats))

print("書き出したファイル：")
for p, st in written:
    print(f" - {p}  (PROF_CH4ブロック: {st['blocks']} / 置換要素数: {st['values_replaced']})")


書き出したファイル：
 - E:\permian_basin\out_prof_ch4\sample_PROF_CH4_1.70.json  (PROF_CH4ブロック: 1 / 置換要素数: 45)
 - E:\permian_basin\out_prof_ch4\sample_PROF_CH4_1.80.json  (PROF_CH4ブロック: 1 / 置換要素数: 45)
 - E:\permian_basin\out_prof_ch4\sample_PROF_CH4_2.05.json  (PROF_CH4ブロック: 1 / 置換要素数: 45)
 - E:\permian_basin\out_prof_ch4\sample_PROF_CH4_2.10.json  (PROF_CH4ブロック: 1 / 置換要素数: 45)
 - E:\permian_basin\out_prof_ch4\sample_PROF_CH4_2.15.json  (PROF_CH4ブロック: 1 / 置換要素数: 45)
 - E:\permian_basin\out_prof_ch4\sample_PROF_CH4_2.20.json  (PROF_CH4ブロック: 1 / 置換要素数: 45)
 - E:\permian_basin\out_prof_ch4\sample_PROF_CH4_2.25.json  (PROF_CH4ブロック: 1 / 置換要素数: 45)
 - E:\permian_basin\out_prof_ch4\sample_PROF_CH4_2.30.json  (PROF_CH4ブロック: 1 / 置換要素数: 45)
 - E:\permian_basin\out_prof_ch4\sample_PROF_CH4_2.35.json  (PROF_CH4ブロック: 1 / 置換要素数: 45)


In [2]:
# Jupyter で実行OK
from pathlib import Path
import json, copy, re, os

# === 入出力設定 ===
src = Path(r"E:\メタン\2025_HISUI_1_地獄の門\MOD\1.7\sample.json")
out_dir = src.parent / "out_prof_ch4"
out_dir.mkdir(exist_ok=True)

# 出したい11個の値
targets = [0.0, 0.25, 0.50, 0.75, 1.00, 1.25, 1.50, 1.75, 2.00, 2.25, 2.50, 2.75, 3.00, 3.25, 3.50, 3.75, 4.00, 4.25, 4.50, 4.75, 5.00]
def fmt(v: float) -> str:
    # 末尾に入れる数値は小数2桁で固定
    return f"{v:.2f}"

_num_tail = re.compile(r'^(.*?)(\d+(?:\.\d+)?)(\.csv)$', re.IGNORECASE)

def update_csv_name(old: str, v: float) -> str:
    """
    例: 'ch_h2ostr_1.50.csv' -> 'ch_h2ostr_1.60.csv'
    末尾の数値だけ置換。無ければ '_1.60' を追記。
    """
    m = _num_tail.match(old)
    if m:
        return f"{m.group(1)}{fmt(v)}{m.group(3)}"
    base, ext = os.path.splitext(old)
    return f"{base}_{fmt(v)}{ext or '.csv'}"

def rewrite_prof_ch4_and_csv(node, new_val: float, inside_prof: bool, stats: dict):
    """
    - inside_prof=True の範囲（= PROF_CH4 サブツリー）では：
        * PROFILE の 1.8 を new_val に置換
          （全要素が1.8なら丸ごと new_val で埋め直し）
        * CSVPRNT（文字列）を末尾数値だけ new_val に更新
    - さらにツリー全体を通して FILEOPTIONS.CSVPRNT も new_val に更新
    """
    if isinstance(node, dict):
        now_inside = inside_prof or (node.get("TYPE") == "PROF_CH4")
        if node.get("TYPE") == "PROF_CH4":
            stats["prof_blocks"] += 1

        # === PROF_CH4サブツリー内の処理 ===
        if now_inside:
            # PROFILE の置換
            if "PROFILE" in node and isinstance(node["PROFILE"], list):
                prof = node["PROFILE"]
                replaced = []
                all18 = len(prof) > 0
                changed = 0
                for x in prof:
                    try:
                        xv = float(x)
                        if abs(xv - 1.8) < 1e-12:
                            replaced.append(new_val)
                            changed += 1
                        else:
                            replaced.append(x)
                            if abs(xv - 1.8) >= 1e-12:
                                all18 = False
                    except Exception:
                        replaced.append(x)
                        all18 = False
                if all18:
                    replaced = [new_val] * len(prof)
                    changed = len(prof)
                node["PROFILE"] = replaced
                stats["profile_vals_replaced"] += changed

            # サブツリー内の CSVPRNT を更新
            if "CSVPRNT" in node and isinstance(node["CSVPRNT"], str):
                node["CSVPRNT"] = update_csv_name(node["CSVPRNT"], new_val)
                stats["csv_prof_updated"] += 1

        # === FILEOPTIONS.CSVPRNT もグローバルに更新 ===
        if "FILEOPTIONS" in node and isinstance(node["FILEOPTIONS"], dict):
            fo = node["FILEOPTIONS"]
            if "CSVPRNT" in fo and isinstance(fo["CSVPRNT"], str):
                fo["CSVPRNT"] = update_csv_name(fo["CSVPRNT"], new_val)
                stats["csv_fileoptions_updated"] += 1

        # 子へ（now_inside を引き継ぎ）
        for k, v in list(node.items()):
            node[k] = rewrite_prof_ch4_and_csv(v, new_val, now_inside, stats)
        return node

    elif isinstance(node, list):
        return [rewrite_prof_ch4_and_csv(v, new_val, inside_prof, stats) for v in node]
    else:
        return node


# === 実行 ===
with src.open("r", encoding="utf-8") as f:
    original = json.load(f)

written = []
for val in targets:
    data = copy.deepcopy(original)
    stats = {
        "prof_blocks": 0,
        "profile_vals_replaced": 0,
        "csv_prof_updated": 0,
        "csv_fileoptions_updated": 0,
    }
    data = rewrite_prof_ch4_and_csv(data, val, inside_prof=False, stats=stats)

    out_path = out_dir / f"{src.stem}_PROF_CH4_{fmt(val)}.json"
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    written.append((out_path, stats))

print("書き出したファイル：")
for p, st in written:
    print(f" - {p}")
    print(f"    PROF_CH4ブロック: {st['prof_blocks']}, PROFILE置換数: {st['profile_vals_replaced']}, "
          f"CSV(PRF): {st['csv_prof_updated']}, CSV(FILEOPTIONS): {st['csv_fileoptions_updated']}")


書き出したファイル：
 - E:\メタン\2025_HISUI_1_地獄の門\MOD\1.7\out_prof_ch4\sample_PROF_CH4_0.00.json
    PROF_CH4ブロック: 1, PROFILE置換数: 45, CSV(PRF): 0, CSV(FILEOPTIONS): 1
 - E:\メタン\2025_HISUI_1_地獄の門\MOD\1.7\out_prof_ch4\sample_PROF_CH4_0.25.json
    PROF_CH4ブロック: 1, PROFILE置換数: 45, CSV(PRF): 0, CSV(FILEOPTIONS): 1
 - E:\メタン\2025_HISUI_1_地獄の門\MOD\1.7\out_prof_ch4\sample_PROF_CH4_0.50.json
    PROF_CH4ブロック: 1, PROFILE置換数: 45, CSV(PRF): 0, CSV(FILEOPTIONS): 1
 - E:\メタン\2025_HISUI_1_地獄の門\MOD\1.7\out_prof_ch4\sample_PROF_CH4_0.75.json
    PROF_CH4ブロック: 1, PROFILE置換数: 45, CSV(PRF): 0, CSV(FILEOPTIONS): 1
 - E:\メタン\2025_HISUI_1_地獄の門\MOD\1.7\out_prof_ch4\sample_PROF_CH4_1.00.json
    PROF_CH4ブロック: 1, PROFILE置換数: 45, CSV(PRF): 0, CSV(FILEOPTIONS): 1
 - E:\メタン\2025_HISUI_1_地獄の門\MOD\1.7\out_prof_ch4\sample_PROF_CH4_1.25.json
    PROF_CH4ブロック: 1, PROFILE置換数: 45, CSV(PRF): 0, CSV(FILEOPTIONS): 1
 - E:\メタン\2025_HISUI_1_地獄の門\MOD\1.7\out_prof_ch4\sample_PROF_CH4_1.50.json
    PROF_CH4ブロック: 1, PROFILE置換数: 45, CSV(PRF